# Day 10 · 数据清洗实战(W2 第 3 天)

目标:识别并处理真实数据里的四大脏东西:重复值、缺失值、类型错乱、异常值。

今日节奏:40min 学习 + 15min 动手 + 5min 自检。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("环境 OK!开始今天的练习 →")


## 任务 0:先造一份"脏数据"

真实数据很少干净。下面这份表故意埋了六种常见问题,后面逐个清掉:

- 名字有空格、大小写不一致、重复行
- age 是字符串、有 "abc" 和异常大的 "999"
- 城市大小写/空格不统一、有空值
- score 有 9999 这种明显打错的异常值


In [ ]:
dirty = pd.DataFrame({
    "name": [" alice ", "Bob", "bob", "Carol", None, " dave ", "Eve", "eve"],
    "age": ["25", "30", "30", "abc", "28", None, "999", "27"],
    "city": ["Beijing", "beijing", " BEIJING ", "Shanghai", "Shanghai", None, "Shenzhen", "Shenzhen"],
    "score": [88, 76, 76, 9999, 91, 82, 79, 85],
})
print(dirty)


## 任务 1:体检(先看清楚病在哪)

- `isna().sum()` → 缺失
- `duplicated().sum()` → 重复行
- `dtypes` → 类型是否对
- `unique()` → 文本取值有没有大小写/空格差异


In [ ]:
print("缺失:\n", dirty.isna().sum())
print("重复行:", dirty.duplicated().sum())
print("类型:\n", dirty.dtypes)
print("city 的取值:", dirty["city"].dropna().unique().tolist())


## 任务 2:去重

`drop_duplicates()` 按整行判断;`subset=["name", "city"]` 只看指定列;`keep="first"` 保留第一条。


In [ ]:
d2 = dirty.drop_duplicates().copy()
print("整行去重后:", d2.shape)
print(d2)


## 任务 3:文本规范化

`.str.strip()` 去首尾空格、`.str.lower()` 统一小写、`.str.title()` 首字母大写。
清洗后同一城市应该只有一个写法,之后再按列分组统计才不会"分裂"。


In [ ]:
d2["name"] = d2["name"].fillna("unknown").str.strip().str.lower()
d2["city"] = d2["city"].fillna("unknown").str.strip().str.lower()
print("city 的取值(清洗后):", d2["city"].unique().tolist())


## 任务 4:类型转换

`pd.to_numeric(列, errors="coerce")` 把字符串转数字,转不了的变成 NaN——然后再按缺失值处理。


In [ ]:
d2["age"] = pd.to_numeric(d2["age"], errors="coerce")
print("转换后:\n", d2)
print("age 缺失数:", d2["age"].isna().sum())


## 任务 5:异常值

两步:
1. 用常识规则框定合法范围(年龄 0~120、分数 0~100),越界的先置 NaN
2. 再按缺失值补齐(中位数),或 `clip(lower, upper)` 硬截断


In [ ]:
d2.loc[d2["age"] > 120, "age"] = np.nan
d2.loc[d2["score"] > 100, "score"] = np.nan

d2["age"] = d2["age"].fillna(d2["age"].median())
d2["score"] = d2["score"].fillna(d2["score"].median())
print("清理完成:\n", d2)
print("剩余缺失:", d2.isna().sum().sum())


## 任务 6:小挑战 🔥(把清洗装进流水线)

写一个 `clean(df)` 函数,把"去重 → 文本规范化 → 类型转换 → 异常值 → 补缺失"串起来,每步打印形状或结果。以后拿到新数据,直接调它。


In [ ]:
def clean(df):
    print("初始:", df.shape)
    df = df.copy()
    df["name"] = df["name"].fillna("unknown").str.strip().str.lower()
    df["city"] = df["city"].fillna("unknown").str.strip().str.lower()
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[df["age"] > 120, "age"] = np.nan
    df.loc[df["score"] > 100, "score"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())
    df["score"] = df["score"].fillna(df["score"].median())
    df = df.drop_duplicates()
    print("清洗后:", df.shape, "| 缺失:", df.isna().sum().sum())
    return df

clean(dirty)


## 自检清单(5 问,答不上就回看今天的格子)

1. 四大类脏数据? → 重复值、缺失值、类型错乱、异常值
2. 文本不一致的典型表现? → 大小写、首尾空格、同义不同写法
3. `pd.to_numeric(errors="coerce")` 干什么? → 转数字,转不了的置 NaN
4. 异常值两步处理? → 先按规则置 NaN,再 fillna(中位数)或 clip 截断
5. 为什么清洗要做成流水线函数? → 可复用、每步可检查,新数据直接套

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day10: data cleaning"; git push
```

然后跟助手说"生成日志"。
